# 🎙️ SoundSense AI: Advanced Spectrogram Analysis & Speech Emotion Recognition
**Author:** Shashank Suthrave  
**Course:** Mini Project / Major Project  
**Domain:** Digital Signal Processing (DSP) & Deep Learning Computer Vision  
**Core Architecture:** 4-Stage 2D Deep CNN (`EmotionCNN`) on Log-Mel Spectrograms with Grad-CAM Explainable AI (XAI)

In [ ]:
# Import Core Libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sounddevice as sd
import soundfile as sf
import librosa
import librosa.display
import torch
from IPython.display import Audio, display

# Import Custom Modular Architecture
from core.audio_processor import load_audio, record_live_audio, resample_for_comparison
from core.feature_extractor import (
    extract_mel_spectrogram,
    compute_fft,
    compute_linear_spectrogram,
    compute_mfcc_suite,
    compute_chroma,
    compute_spectral_features,
    extract_dsp_summary_table,
    compare_sample_rates
)
from core.model import EmotionPredictor, DEFAULT_EMOTIONS, EMOTION_EMOJIS

print("Environment initialized successfully! PyTorch version:", torch.__version__)

## 1. Audio Ingestion & Voice Activity Detection (VAD)
Load existing project audio or record a new voice sample with automatic silence trimming and amplitude normalization.

In [ ]:
audio_path = "audio_data1.wav"
y, sr = load_audio(audio_path, sample_rate=22050, trim_silence=True)

print(f"Audio File:        {audio_path}")
print(f"Sampling Rate:     {sr} Hz")
print(f"Total Duration:    {len(y) / sr:.2f} seconds")
print(f"Total Samples:     {len(y):,}")

# Playback in Notebook
Audio(y, rate=sr)

## 2. Advanced Time-Frequency Feature Extraction
Extract 2D Log-Mel Spectrogram (128x128), FFT magnitude spectrum, 13 MFCCs with Velocity & Acceleration Deltas, 12-pitch Chroma, and Spectral Moments.

In [ ]:
# 1. 2D Normalized Log-Mel Spectrogram for CNN Input (128, 128)
mel_2d = extract_mel_spectrogram(y, sr=sr, target_frames=128)

# 2. Fast Fourier Transform (FFT) Magnitude
freq_axis, fft_mag = compute_fft(y, sr)

# 3. MFCC Suite (Static + Delta Velocity + Delta-Delta Acceleration)
mfcc_dict = compute_mfcc_suite(y, sr, n_mfcc=13)

# 4. 12-Semitone Chroma STFT
chroma = compute_chroma(y, sr)

# 5. Spectral Moments (Centroid, Bandwidth, Roll-off, ZCR, RMS)
spectral = compute_spectral_features(y, sr)

print(f"Mel-Spectrogram Shape: {mel_2d.shape} | Range: [{np.min(mel_2d):.2f}, {np.max(mel_2d):.2f}]")
print(f"MFCC Matrix Shape:     {mfcc_dict['mfcc'].shape}")
print(f"Chroma Matrix Shape:   {chroma.shape}")

## 3. Comprehensive Signal Visualization Dashboard
Plot all acoustic features side-by-side.

In [ ]:
plt.figure(figsize=(15, 10))

# Subplot 1: Waveform
plt.subplot(3, 2, 1)
time_axis = np.linspace(0, len(y) / sr, len(y))
plt.plot(time_axis, y, color='dodgerblue', alpha=0.85)
plt.title("Time-Domain Waveform & Envelope")
plt.xlabel("Time (s)")
plt.ylabel("Normalized Amplitude")

# Subplot 2: FFT Magnitude Spectrum
plt.subplot(3, 2, 2)
plt.plot(freq_axis, fft_mag, color='indigo')
plt.xlim(0, min(8000, sr // 2))
plt.title("FFT Frequency Magnitude Spectrum")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")

# Subplot 3: 2D Log-Mel Spectrogram
plt.subplot(3, 2, 3)
plt.imshow(mel_2d, origin='lower', aspect='auto', cmap='inferno')
plt.colorbar(label="Log-Mel Energy")
plt.title("2D Normalized Log-Mel Spectrogram (128x128)")
plt.xlabel("Normalized Time Frames")
plt.ylabel("Mel Filterbank")

# Subplot 4: MFCCs (13 coefficients)
plt.subplot(3, 2, 4)
librosa.display.specshow(mfcc_dict["mfcc"], sr=sr, x_axis='time', cmap='plasma')
plt.colorbar(label="Coefficients")
plt.title("Mel-Frequency Cepstral Coefficients (MFCCs)")
plt.xlabel("Time (s)")
plt.ylabel("MFCC Order")

# Subplot 5: 12-Pitch Chroma Profile
plt.subplot(3, 2, 5)
pitches = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
plt.imshow(chroma, origin='lower', aspect='auto', cmap='viridis')
plt.yticks(ticks=np.arange(12), labels=pitches)
plt.colorbar(label="Energy")
plt.title("12-Semitone Chroma STFT (Harmonic Profile)")
plt.xlabel("Frames")

# Subplot 6: Spectral Centroid vs Roll-off
plt.subplot(3, 2, 6)
plt.plot(spectral["centroid"][0], label="Spectral Centroid", color="teal")
plt.plot(spectral["rolloff"][0], label="Spectral Roll-off (85%)", color="coral", linestyle="--")
plt.title("Spectral Moments Over Time")
plt.xlabel("Frames")
plt.ylabel("Frequency (Hz)")
plt.legend()

plt.tight_layout()
plt.show()

## 4. Acoustic Statistical Summary Table
Provides Mean, Standard Deviation, Minimum, and Maximum values across time frames, avoiding the loss of information caused by naive 1D averaging.

In [ ]:
df_summary = extract_dsp_summary_table(y, sr)
display(df_summary)

## 5. Multi-Sample Rate Comparison Study (8kHz vs 16kHz vs 22.05kHz vs 44.1kHz)
Elevating the original experiment from `project.ipynb` to evaluate the impact of the Nyquist cut-off.

In [ ]:
test_rates = [8000, 16000, 22050, 44100]
df_comparison = compare_sample_rates(y, sr, sample_rates=test_rates)
display(df_comparison)

# Plot comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar([str(r) for r in df_comparison['Sample Rate (Hz)']], df_comparison['Centroid (Hz)'], color='steelblue')
plt.title("Spectral Centroid vs Sample Rate")
plt.xlabel("Sample Rate (Hz)")
plt.ylabel("Centroid (Hz)")

plt.subplot(1, 2, 2)
plt.bar([str(r) for r in df_comparison['Sample Rate (Hz)']], df_comparison['Rolloff (Hz)'], color='salmon')
plt.title("Spectral Roll-off vs Sample Rate")
plt.xlabel("Sample Rate (Hz)")
plt.ylabel("Roll-off (Hz)")
plt.tight_layout()
plt.show()

## 6. Deep Learning 2D CNN Speech Emotion Recognition (SER)
Run inference using our trained 2D Convolutional Neural Network.

In [ ]:
# Initialize Predictor
weights_path = os.path.join("models", "emotion_cnn.pt")
predictor = EmotionPredictor(model_path=weights_path if os.path.exists(weights_path) else None)

# Run Inference with Grad-CAM computation
pred = predictor.predict(mel_2d, compute_gradcam=True)

print("=" * 50)
print(f"🎯 PREDICTED EMOTION: {pred['emoji']} {pred['predicted_emotion']}")
print(f"📊 CONFIDENCE SCORE:  {pred['confidence']}%")
print(f"🌐 VALENCE / AROUSAL: {pred['valence_arousal']}")
print("=" * 50)
print("\nClass Probability Distribution:")
for emo, prob in pred['probabilities'].items():
    print(f"  {EMOTION_EMOJIS.get(emo, '')} {emo:<10}: {prob:>5.2f}%")

## 7. Explainable AI: Grad-CAM Spectrogram Attention Map
Inspect which frequency-time bins the convolutional network focused on to classify the emotion.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(mel_2d, origin='lower', aspect='auto', cmap='inferno')
plt.title("Input Log-Mel Spectrogram")
plt.xlabel("Time Frames")
plt.ylabel("Mel Filterbanks")
plt.colorbar(label="Normalized Log-Mel Energy")

plt.subplot(1, 2, 2)
plt.imshow(pred["attention_map"], origin='lower', aspect='auto', cmap='jet')
plt.title(f"Grad-CAM Attention Heatmap for Class '{pred['predicted_emotion']}'")
plt.xlabel("Time Frames")
plt.ylabel("Mel Filterbanks")
plt.colorbar(label="Gradient Activation")

plt.tight_layout()
plt.show()